# Análisis del Progreso Mundial de Vacunación

Este notebook analiza el progreso de vacunación contra COVID-19 a nivel mundial.

## Objetivos
- Cargar y procesar datos de vacunación mundial
- Analizar tendencias de vacunación por país y región
- Visualizar el progreso de vacunación
- Identificar países líderes y rezagados
- Generar insights y conclusiones

## 1. Instalación de Dependencias

In [ ]:
# Instalar librerías necesarias
!pip install pandas numpy matplotlib seaborn plotly -q

## 2. Importar Librerías

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from datetime import datetime
import warnings

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

print("✓ Librerías importadas correctamente")

## 3. Cargar Datos de Vacunación

Utilizamos datos de Our World in Data (OWID), una fuente confiable de datos de vacunación COVID-19.

In [ ]:
# Cargar datos desde Our World in Data
url = 'https://raw.githubusercontent.com/owid/covid-19-data/master/public/data/vaccinations/vaccinations.csv'

try:
    df = pd.read_csv(url)
    print(f"✓ Datos cargados exitosamente")
    print(f"  Dimensiones: {df.shape[0]} filas x {df.shape[1]} columnas")
    print(f"  Países: {df['location'].nunique()}")
    print(f"  Período: {df['date'].min()} a {df['date'].max()}")
except Exception as e:
    print(f"Error al cargar datos: {e}")
    # Crear datos de ejemplo si no se puede acceder a la fuente
    print("Generando datos de ejemplo...")
    df = pd.DataFrame({
        'location': ['Argentina', 'Brasil', 'Chile', 'Colombia', 'México'] * 100,
        'date': pd.date_range('2021-01-01', periods=100).tolist() * 5,
        'total_vaccinations': np.cumsum(np.random.randint(1000, 10000, 500)),
        'people_vaccinated': np.cumsum(np.random.randint(500, 5000, 500)),
        'people_fully_vaccinated': np.cumsum(np.random.randint(300, 3000, 500)),
        'total_vaccinations_per_hundred': np.random.uniform(0, 100, 500)
    })

## 4. Exploración de Datos

In [ ]:
# Mostrar primeras filas
print("Primeras filas del dataset:")
display(df.head(10))

In [ ]:
# Información del dataset
print("Información del dataset:")
df.info()

In [ ]:
# Estadísticas descriptivas
print("Estadísticas descriptivas:")
display(df.describe())

In [ ]:
# Columnas disponibles
print("Columnas disponibles:")
for i, col in enumerate(df.columns, 1):
    print(f"{i}. {col}")

## 5. Limpieza y Preparación de Datos

In [ ]:
# Convertir fecha a formato datetime
df['date'] = pd.to_datetime(df['date'])

# Verificar valores nulos
print("Valores nulos por columna:")
null_counts = df.isnull().sum()
null_percentages = (null_counts / len(df)) * 100
null_df = pd.DataFrame({
    'Nulos': null_counts,
    'Porcentaje': null_percentages.round(2)
})
display(null_df[null_df['Nulos'] > 0].sort_values('Nulos', ascending=False))

In [ ]:
# Filtrar solo países (excluir agregaciones regionales)
exclude_locations = ['World', 'Europe', 'Asia', 'Africa', 'North America', 'South America', 
                     'European Union', 'High income', 'Low income', 'Lower middle income', 
                     'Upper middle income', 'Oceania']

df_countries = df[~df['location'].isin(exclude_locations)].copy()
print(f"✓ Filtrado realizado: {df_countries['location'].nunique()} países")

## 6. Análisis: Datos Más Recientes por País

In [ ]:
# Obtener el registro más reciente de cada país
df_latest = df_countries.sort_values('date').groupby('location').last().reset_index()

print(f"Datos más recientes (fecha: {df_latest['date'].max().strftime('%Y-%m-%d')}):")
print(f"Total de países con datos: {len(df_latest)}")

## 7. Top 20 Países con Mayor Cobertura de Vacunación

In [ ]:
# Top países por vacunación per cápita
if 'total_vaccinations_per_hundred' in df_latest.columns:
    top_countries = df_latest.nlargest(20, 'total_vaccinations_per_hundred')[['location', 'total_vaccinations_per_hundred', 'people_fully_vaccinated_per_hundred']]
    
    print("Top 20 países por cobertura de vacunación:")
    display(top_countries.reset_index(drop=True))
    
    # Visualización
    fig, ax = plt.subplots(figsize=(12, 8))
    
    countries = top_countries['location'].values
    values = top_countries['total_vaccinations_per_hundred'].values
    
    bars = ax.barh(countries, values, color='steelblue')
    ax.set_xlabel('Dosis por cada 100 personas', fontsize=12)
    ax.set_title('Top 20 Países con Mayor Cobertura de Vacunación', fontsize=14, fontweight='bold')
    ax.invert_yaxis()
    
    # Añadir valores en las barras
    for i, bar in enumerate(bars):
        width = bar.get_width()
        ax.text(width, bar.get_y() + bar.get_height()/2, 
                f'{width:.1f}',
                ha='left', va='center', fontsize=9, fontweight='bold')
    
    plt.tight_layout()
    plt.show()
else:
    print("Columna 'total_vaccinations_per_hundred' no disponible")

## 8. Análisis de Progreso Temporal

In [ ]:
# Seleccionar países de interés para análisis temporal
countries_of_interest = ['United States', 'United Kingdom', 'Israel', 'Chile', 'Argentina', 
                         'Brazil', 'Germany', 'France', 'Spain', 'Italy']

# Filtrar países disponibles
available_countries = [c for c in countries_of_interest if c in df_countries['location'].values]

if available_countries:
    df_temporal = df_countries[df_countries['location'].isin(available_countries)].copy()
    
    # Visualizar progreso temporal
    if 'people_fully_vaccinated_per_hundred' in df_temporal.columns:
        fig = px.line(df_temporal, 
                      x='date', 
                      y='people_fully_vaccinated_per_hundred',
                      color='location',
                      title='Progreso de Vacunación Completa por País',
                      labels={'people_fully_vaccinated_per_hundred': 'Personas completamente vacunadas (%)',
                             'date': 'Fecha',
                             'location': 'País'})
        fig.update_layout(height=600, hovermode='x unified')
        fig.show()
    else:
        print("Columna de vacunación completa no disponible")
else:
    print("Países de interés no disponibles en el dataset")

## 9. Distribución Global de Vacunación

In [ ]:
# Histograma de distribución
if 'total_vaccinations_per_hundred' in df_latest.columns:
    data_clean = df_latest[df_latest['total_vaccinations_per_hundred'].notna()]['total_vaccinations_per_hundred']
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
    
    # Histograma
    ax1.hist(data_clean, bins=30, color='skyblue', edgecolor='black', alpha=0.7)
    ax1.set_xlabel('Dosis por cada 100 personas', fontsize=12)
    ax1.set_ylabel('Número de países', fontsize=12)
    ax1.set_title('Distribución de Cobertura de Vacunación', fontsize=14, fontweight='bold')
    ax1.axvline(data_clean.mean(), color='red', linestyle='--', linewidth=2, label=f'Media: {data_clean.mean():.1f}')
    ax1.axvline(data_clean.median(), color='green', linestyle='--', linewidth=2, label=f'Mediana: {data_clean.median():.1f}')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Box plot
    ax2.boxplot(data_clean, vert=True, patch_artist=True,
                boxprops=dict(facecolor='lightblue', alpha=0.7),
                medianprops=dict(color='red', linewidth=2))
    ax2.set_ylabel('Dosis por cada 100 personas', fontsize=12)
    ax2.set_title('Box Plot de Cobertura de Vacunación', fontsize=14, fontweight='bold')
    ax2.grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.show()
    
    # Estadísticas
    print("\nEstadísticas de cobertura de vacunación:")
    print(f"Media: {data_clean.mean():.2f} dosis/100 personas")
    print(f"Mediana: {data_clean.median():.2f} dosis/100 personas")
    print(f"Desviación estándar: {data_clean.std():.2f}")
    print(f"Mínimo: {data_clean.min():.2f}")
    print(f"Máximo: {data_clean.max():.2f}")

## 10. Análisis por Velocidad de Vacunación

In [ ]:
# Calcular velocidad de vacunación (últimos 30 días)
if 'daily_vaccinations' in df_countries.columns:
    recent_date = df_countries['date'].max()
    date_30_days_ago = recent_date - pd.Timedelta(days=30)
    
    df_recent = df_countries[df_countries['date'] >= date_30_days_ago].copy()
    
    # Promedio de vacunaciones diarias por país
    avg_daily = df_recent.groupby('location')['daily_vaccinations'].mean().sort_values(ascending=False)
    
    print("Top 10 países por velocidad de vacunación (últimos 30 días):")
    top_10_speed = avg_daily.head(10)
    
    for i, (country, value) in enumerate(top_10_speed.items(), 1):
        print(f"{i}. {country}: {value:,.0f} dosis/día")
    
    # Visualización
    fig, ax = plt.subplots(figsize=(12, 6))
    top_10_speed.plot(kind='bar', ax=ax, color='coral')
    ax.set_ylabel('Vacunaciones diarias promedio', fontsize=12)
    ax.set_title('Top 10 Países por Velocidad de Vacunación (últimos 30 días)', 
                 fontsize=14, fontweight='bold')
    ax.set_xlabel('País', fontsize=12)
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()
else:
    print("Datos de vacunación diaria no disponibles")

## 11. Mapa Mundial de Vacunación

In [ ]:
# Crear mapa coroplético mundial
if 'total_vaccinations_per_hundred' in df_latest.columns:
    # Usar códigos ISO para el mapa
    if 'iso_code' in df_latest.columns:
        fig = px.choropleth(df_latest,
                           locations='iso_code',
                           color='total_vaccinations_per_hundred',
                           hover_name='location',
                           hover_data={'total_vaccinations_per_hundred': ':.2f',
                                      'people_fully_vaccinated_per_hundred': ':.2f'},
                           color_continuous_scale='RdYlGn',
                           title='Cobertura de Vacunación Mundial (Dosis por 100 personas)',
                           labels={'total_vaccinations_per_hundred': 'Dosis/100 personas'})
        fig.update_layout(height=600)
        fig.show()
    else:
        print("Códigos ISO no disponibles para crear el mapa")
else:
    print("Datos de vacunación per cápita no disponibles")

## 12. Comparación: América Latina

In [ ]:
# Países de América Latina
latam_countries = ['Argentina', 'Bolivia', 'Brazil', 'Chile', 'Colombia', 'Costa Rica',
                   'Cuba', 'Dominican Republic', 'Ecuador', 'El Salvador', 'Guatemala',
                   'Honduras', 'Mexico', 'Nicaragua', 'Panama', 'Paraguay', 'Peru',
                   'Uruguay', 'Venezuela']

# Filtrar países disponibles
latam_available = [c for c in latam_countries if c in df_latest['location'].values]

if latam_available and 'total_vaccinations_per_hundred' in df_latest.columns:
    df_latam = df_latest[df_latest['location'].isin(latam_available)].copy()
    df_latam = df_latam.sort_values('total_vaccinations_per_hundred', ascending=False)
    
    print("Cobertura de vacunación en América Latina:")
    display(df_latam[['location', 'total_vaccinations_per_hundred', 'people_fully_vaccinated_per_hundred']].reset_index(drop=True))
    
    # Visualización
    fig, ax = plt.subplots(figsize=(12, 8))
    
    x = np.arange(len(df_latam))
    width = 0.35
    
    if 'people_fully_vaccinated_per_hundred' in df_latam.columns:
        ax.bar(x - width/2, df_latam['total_vaccinations_per_hundred'], width, 
               label='Total dosis', alpha=0.8, color='steelblue')
        ax.bar(x + width/2, df_latam['people_fully_vaccinated_per_hundred'], width,
               label='Completamente vacunados', alpha=0.8, color='coral')
    else:
        ax.bar(x, df_latam['total_vaccinations_per_hundred'], 
               label='Total dosis', alpha=0.8, color='steelblue')
    
    ax.set_ylabel('Porcentaje (%)', fontsize=12)
    ax.set_title('Cobertura de Vacunación en América Latina', fontsize=14, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(df_latam['location'], rotation=45, ha='right')
    ax.legend()
    ax.grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.show()
else:
    print("Datos de América Latina no disponibles")

## 13. Análisis de Correlaciones

In [ ]:
# Seleccionar columnas numéricas relevantes
numeric_cols = df_latest.select_dtypes(include=[np.number]).columns.tolist()

# Remover columnas no relevantes
cols_to_exclude = ['date']
numeric_cols = [col for col in numeric_cols if col not in cols_to_exclude]

if len(numeric_cols) > 1:
    # Calcular matriz de correlación
    correlation_matrix = df_latest[numeric_cols].corr()
    
    # Visualizar matriz de correlación
    fig, ax = plt.subplots(figsize=(12, 10))
    sns.heatmap(correlation_matrix, annot=True, fmt='.2f', cmap='coolwarm', 
                center=0, square=True, linewidths=1, ax=ax,
                cbar_kws={"shrink": 0.8})
    ax.set_title('Matriz de Correlación - Variables de Vacunación', 
                 fontsize=14, fontweight='bold', pad=20)
    plt.tight_layout()
    plt.show()
else:
    print("No hay suficientes columnas numéricas para análisis de correlación")

## 14. Resumen Ejecutivo

In [ ]:
# Generar resumen de hallazgos clave
print("="*80)
print(" " * 25 + "RESUMEN EJECUTIVO")
print("="*80)
print()

# Estadísticas generales
print("📊 ESTADÍSTICAS GENERALES:")
print(f"  • Total de países analizados: {df_latest['location'].nunique()}")
print(f"  • Última actualización: {df_latest['date'].max().strftime('%Y-%m-%d')}")
print(f"  • Período de análisis: {df_countries['date'].min().strftime('%Y-%m-%d')} a {df_countries['date'].max().strftime('%Y-%m-%d')}")
print()

# Cobertura de vacunación
if 'total_vaccinations_per_hundred' in df_latest.columns:
    data_clean = df_latest[df_latest['total_vaccinations_per_hundred'].notna()]['total_vaccinations_per_hundred']
    print("💉 COBERTURA DE VACUNACIÓN:")
    print(f"  • Promedio mundial: {data_clean.mean():.2f} dosis por 100 personas")
    print(f"  • Mediana mundial: {data_clean.median():.2f} dosis por 100 personas")
    print(f"  • País con mayor cobertura: {df_latest.loc[df_latest['total_vaccinations_per_hundred'].idxmax(), 'location']} ({df_latest['total_vaccinations_per_hundred'].max():.2f})")
    print(f"  • País con menor cobertura: {df_latest.loc[df_latest['total_vaccinations_per_hundred'].idxmin(), 'location']} ({df_latest['total_vaccinations_per_hundred'].min():.2f})")
    print()

# Vacunación completa
if 'people_fully_vaccinated_per_hundred' in df_latest.columns:
    data_full = df_latest[df_latest['people_fully_vaccinated_per_hundred'].notna()]['people_fully_vaccinated_per_hundred']
    print("✅ VACUNACIÓN COMPLETA:")
    print(f"  • Promedio mundial: {data_full.mean():.2f}% de población completamente vacunada")
    print(f"  • Mediana mundial: {data_full.median():.2f}% de población completamente vacunada")
    print()

# Top 5 países
if 'total_vaccinations_per_hundred' in df_latest.columns:
    top_5 = df_latest.nlargest(5, 'total_vaccinations_per_hundred')
    print("🏆 TOP 5 PAÍSES CON MAYOR COBERTURA:")
    for i, (idx, row) in enumerate(top_5.iterrows(), 1):
        print(f"  {i}. {row['location']}: {row['total_vaccinations_per_hundred']:.2f} dosis/100 personas")
    print()

# América Latina
if latam_available and 'total_vaccinations_per_hundred' in df_latest.columns:
    df_latam = df_latest[df_latest['location'].isin(latam_available)].copy()
    latam_avg = df_latam['total_vaccinations_per_hundred'].mean()
    print("🌎 AMÉRICA LATINA:")
    print(f"  • Promedio regional: {latam_avg:.2f} dosis por 100 personas")
    print(f"  • Líder regional: {df_latam.loc[df_latam['total_vaccinations_per_hundred'].idxmax(), 'location']}")
    print()

print("="*80)
print("\n📝 CONCLUSIONES:")
print("  1. Existe una gran disparidad en la cobertura de vacunación entre países")
print("  2. Los países desarrollados generalmente muestran mayor cobertura")
print("  3. América Latina muestra progreso significativo pero heterogéneo")
print("  4. La velocidad de vacunación varía considerablemente por región")
print("  5. Se requiere mayor esfuerzo para alcanzar cobertura universal")
print("="*80)

## 15. Exportar Resultados

In [ ]:
# Exportar datos procesados
try:
    # Guardar datos más recientes
    df_latest.to_csv('vacunacion_mundial_reciente.csv', index=False)
    print("✓ Datos exportados a 'vacunacion_mundial_reciente.csv'")
    
    # Guardar top países
    if 'total_vaccinations_per_hundred' in df_latest.columns:
        top_countries = df_latest.nlargest(50, 'total_vaccinations_per_hundred')
        top_countries.to_csv('top_50_paises_vacunacion.csv', index=False)
        print("✓ Top 50 países exportados a 'top_50_paises_vacunacion.csv'")
except Exception as e:
    print(f"Error al exportar: {e}")

## Conclusión

Este análisis proporciona una visión completa del progreso mundial de vacunación contra COVID-19. Los datos muestran:

- **Progreso significativo**: Muchos países han logrado altas tasas de vacunación
- **Desigualdad global**: Persiste una brecha importante entre países desarrollados y en desarrollo
- **Tendencias regionales**: Algunas regiones muestran avances notables mientras otras enfrentan desafíos
- **Velocidad de vacunación**: La capacidad de administración varía considerablemente

### Recomendaciones:
1. Fortalecer la distribución equitativa de vacunas
2. Aumentar la capacidad de administración en países rezagados
3. Continuar monitoreando el progreso y adaptando estrategias
4. Fomentar la cooperación internacional para cobertura universal

---

**Fuente de datos**: Our World in Data (OWID) - COVID-19 Vaccination Dataset  
**Última actualización**: Ver celda de carga de datos  
**Análisis realizado**: 2026